In [4]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# Download stopwords dataset (only needs to be run once)
nltk.download('stopwords', quiet=True)

# --- TASK 1: PREPROCESS A PARAGRAPH ---
paragraph = "Wow! NLP is incredible. I am learning text preprocessing, and it's fun!"
print(f"Original: {paragraph}\n")

# 1. Lowercase
text_lower = paragraph.lower()

# 2. Remove punctuation using Regular Expressions
text_no_punct = re.sub(r'[^\w\s]', '', text_lower)

# 3. Remove stopwords
stop_words = set(stopwords.words('english'))
tokens = text_no_punct.split()
clean_tokens = [word for word in tokens if word not in stop_words]

print(f"Cleaned Tokens: {clean_tokens}\n")
print("-" * 40)

# --- TASK 2: CONVERT CORPUS TO TF-IDF MATRIX ---
# We'll use our cleaned tokens as a single document, plus a second document to compare
corpus = [
    " ".join(clean_tokens),
    "nlp text preprocessing allows machine learning models understand language"
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nTF-IDF Matrix (Dense Array):\n", tfidf_matrix.toarray())

Original: Wow! NLP is incredible. I am learning text preprocessing, and it's fun!

Cleaned Tokens: ['wow', 'nlp', 'incredible', 'learning', 'text', 'preprocessing', 'fun']

----------------------------------------
Vocabulary: ['allows' 'fun' 'incredible' 'language' 'learning' 'machine' 'models'
 'nlp' 'preprocessing' 'text' 'understand' 'wow']

TF-IDF Matrix (Dense Array):
 [[0.         0.44610081 0.44610081 0.         0.3174044  0.
  0.         0.3174044  0.3174044  0.3174044  0.         0.44610081]
 [0.37729199 0.         0.         0.37729199 0.26844636 0.37729199
  0.37729199 0.26844636 0.26844636 0.26844636 0.37729199 0.        ]]


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# 1. Create the dummy SMS dataset
data = {
    'message': [
        "Win a FREE iPhone now! Click here", 
        "Hey, what time is the meeting?", 
        "URGENT: Your bank account is compromised. Call 1-800", 
        "Can you pick up milk on the way home?",
        "Congratulations! You've been selected for a cash prize",
        "See you at dinner tonight!"
    ],
    'label': ['spam', 'ham', 'spam', 'ham', 'spam', 'ham']
}
df = pd.DataFrame(data)

# 2. Split data (Train on 66%, Test on 33%)
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label'], test_size=0.33, random_state=42
)

# 3. Vectorize the text 
vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test) # Note: We use .transform() here, NOT fit_transform()

# 4. Train the Naive Bayes classifier
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_tfidf, y_train)

# 5. Predict on the test set
y_pred = nb_classifier.predict(X_test_tfidf)

# 6. Evaluate (Warning fixed with zero_division=0)
print("--- TASK 4: EVALUATION ---")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

--- TASK 4: EVALUATION ---
Confusion Matrix:
 [[1 0]
 [1 0]]

Classification Report:
               precision    recall  f1-score   support

         ham       0.50      1.00      0.67         1
        spam       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("--- PRACTICE 1: MOVIE SENTIMENT CLASSIFIER ---")
# Dummy Data
reviews = [
    "Absolutely loved this movie! The acting was brilliant.",
    "Terrible plot, awful acting, complete waste of time.",
    "It was a masterpiece from start to finish.",
    "I fell asleep halfway through. Very boring."
]
sentiment_labels = ["positive", "negative", "positive", "negative"]

# Build & Train Pipeline
sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', MultinomialNB())
])
sentiment_pipeline.fit(reviews, sentiment_labels)

# Test it
new_review = ["The cinematography was beautiful and the story was great."]
print(f"Review: '{new_review[0]}'")
print(f"Predicted Sentiment: {sentiment_pipeline.predict(new_review)[0].upper()}\n")


print("-" * 50)


print("--- PRACTICE 2: NEWS CATEGORY CLASSIFIER ---")
# Dummy Data
headlines = [
    "Quarterback throws game-winning touchdown in final seconds",
    "New AI chip released, promising faster processing speeds",
    "Senate passes new infrastructure bill after long debate",
    "Basketball star signs massive contract extension",
    "Tech stocks plummet as new software bug is discovered",
    "Mayor announces new policy for city zoning"
]
news_categories = ["Sports", "Tech", "Politics", "Sports", "Tech", "Politics"]

# Build & Train Pipeline
news_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', LogisticRegression())
])
news_pipeline.fit(headlines, news_categories)

# Test it
test_headlines = [
    "Local team wins the championship game!",
    "The president vetoes the controversial new law.",
    "Startup invents a new smartphone battery."
]
test_actual_labels = ["Sports", "Politics", "Tech"] # What they actually are
test_predictions = news_pipeline.predict(test_headlines)

# Display results
for headline, category in zip(test_headlines, test_predictions):
    print(f"[{category.upper()}] - {headline}")

print("\nNews Evaluation Report:")
print(classification_report(test_actual_labels, test_predictions, zero_division=0))

--- PRACTICE 1: MOVIE SENTIMENT CLASSIFIER ---
Review: 'The cinematography was beautiful and the story was great.'
Predicted Sentiment: NEGATIVE

--------------------------------------------------
--- PRACTICE 2: NEWS CATEGORY CLASSIFIER ---
[SPORTS] - Local team wins the championship game!
[POLITICS] - The president vetoes the controversial new law.
[POLITICS] - Startup invents a new smartphone battery.

News Evaluation Report:
              precision    recall  f1-score   support

    Politics       0.50      1.00      0.67         1
      Sports       1.00      1.00      1.00         1
        Tech       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.50      0.67      0.56         3
weighted avg       0.50      0.67      0.56         3

